In [1]:
from aoespy import *

/home6/afahad/.local/lib/python3.9/site-packages/aoespy.py:19: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
import xarray as xr
import ecco_v4_py as ecco
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
from glob import glob
import os


#

In [4]:
#RP5061=xr.open_mfdataset('/nobackupp27/afahad/exp/IAU_exp/GEOSMIT_RP0506/holding/geosgcm_cape/200505/*cape*200505*z.nc4')
ME5061=xr.open_mfdataset('/nobackupp27/afahad/exp/IAU_exp/GEOSMIT_ME0506/holding/geosgcm_cape/200505/*cape*200505*z.nc4')

In [5]:
ME506s=xr.open_mfdataset('/nobackupp27/afahad/exp/IAU_exp/GEOSMIT_ME0506/holding/geosgcm_surf/200505/*surf*200505*z.nc4')

In [9]:
cape=ME5061.CAPE[2::3].compute()


In [73]:
T=ME5061.T[2::3].compute()

In [75]:
lh=ME506s.LHFX.compute()


In [10]:
omg=ME5061.OMEGA[2::3].sel(lev=500).compute()

In [76]:
sf=ME506s.SHFX.compute()


In [45]:
tsa=ME506s.TA.compute()


In [41]:
ts=ME506s.TS_FOUND.compute()


In [109]:
sw=ME506s.SWGNET.compute()

In [43]:
ts=ts[1:,:,:]-ts[:-1,:,:].data

In [60]:
# lh=lh[1:,:,:]-lh[:-1,:,:].data

In [74]:
# sf=sf[1:,:,:]-sf[:-1,:,:].data

In [124]:
ticks1=[]

for i in range(len(sw.time)):
    ticks1=append(ticks1,str(sw.time.data[i])[5:10])
    

In [1]:
x1 = 143
x2 = 143
y1 = -1
y2 = -1

plt.figure(figsize=(11, 7.5))

plt.subplot(2, 2, 1)
plt.plot((cape).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color='darkblue')
#plt.plot((omg).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color='orange')
plt.title('(a) CAPE',fontsize=12,fontweight='bold')
plt.xticks(arange(len(ticks))[::60], ticks[::60], fontsize=8)
plt.legend(['Reanalysis IC'])
plt.xlabel('Date (month-day)')
plt.ylabel('J kg-1')
plt.grid(True, linestyle='--', alpha=0.6)


ax2_main = plt.subplot(2, 2, 2)
ax2_main.set_title('(b) OMEGA 500 & SW', fontsize=12, fontweight='bold')

# Plot 'sw' (daily) on the first y-axis (left)
color_sw = 'darkblue'
ax2_main.set_xlabel('Date (month-day)')
ax2_main.set_ylabel('Shortwave Radiation (sw)', color=color_sw)
x_daily = np.arange(len(sw.time))
line_sw = ax2_main.plot(x_daily, (sw).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color=color_sw, label='sw (Daily)')
ax2_main.tick_params(axis='y', labelcolor=color_sw)
ax2_main.set_xticks(np.arange(len(ticks1))[::7])
ax2_main.set_xticklabels(ticks1[::7], fontsize=8, rotation=45)
ax2_main.grid(True, linestyle='--', alpha=0.6)
ax2_main.set_ylim(80,350)



# Create a second y-axis that shares the same x-axis
ax2_twin = ax2_main.twinx()

# Plot 'omg' (3-hourly) on the second y-axis (right)
color_omg = 'green'
ax2_twin.set_ylabel('OMEGA 500 (hPa/s)', color=color_omg)
# Scale the x-axis for the 3-hourly data to fit the daily axis
x_3hrly = np.linspace(0, len(sw.time) - 1, len(omg.time))
line_omg = ax2_twin.plot(x_3hrly, (omg).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color=color_omg, label='Omega500 (3-hourly)')
ax2_twin.tick_params(axis='y', labelcolor=color_omg)
ax2_twin.set_ylim(-1,.1)

# Create a combined legend
lines_2 = line_sw + line_omg
labels_2 = [l.get_label() for l in lines_2]
ax2_main.legend(lines_2, labels_2, loc=4)


ax3 = plt.subplot(2, 2, 3)
plt.title('(c) Heat Fluxes', fontsize=12, fontweight='bold')

# Plot 'sf' (Sensible Heat) on the first y-axis (left)
color3 = 'green'
ax3.set_xlabel('Date (month-day)')
ax3.set_ylabel('Sensible Heat Flux (w/m^2)', color=color3)
line3 = ax3.plot((sf).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color=color3, label='Sensible Heat flux')
ax3.tick_params(axis='y', labelcolor=color3)
ax3.set_xticks(np.arange(len(ticks))[::60])
ax3.set_xticklabels(ticks[::60], fontsize=8, rotation=45)
ax3.grid(True, linestyle='--', alpha=0.6)
ax3.set_ylim(2,45)
# Create a second y-axis that shares the same x-axis
ax4 = ax3.twinx()

# Plot 'lh' (Latent Heat) on the second y-axis (right)
color4 = 'darkblue'
ax4.set_ylabel('vertical Heat flux 500 (K*hPa/s)', color=color4)
line4 = ax4.plot((-T.sel(lev=500)*omg.data).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color=color4, label='- T*omega500')
ax4.tick_params(axis='y', labelcolor=color4)
ax4.set_ylim(-100,210)

# Create a combined legend for both lines
lines_3_4 = line3 + line4
labels_3_4 = [l.get_label() for l in lines_3_4]
ax3.legend(lines_3_4, labels_3_4, loc='best')


ax1 = plt.subplot(2, 2, 4)
plt.title('(d) SST tendecies and Air Temperature', fontsize=12, fontweight='bold')

# Plot 'ts' on the first y-axis (left)
color1 = 'green'
ax1.set_xlabel('Date (month-day)')
ax1.set_ylabel('SST tendencies (K/3hr)', color=color1)
line1 = ax1.plot((ts).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color=color1, label='SST tendecies')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(np.arange(len(ticks))[::60])
ax1.set_xticklabels(ticks[::60], fontsize=8, rotation=45)
ax1.grid(True, linestyle='--', alpha=0.6)

# Create a second y-axis that shares the same x-axis
ax2 = ax1.twinx()

# Plot 'tsa' on the second y-axis (right)
color2 = 'darkblue'
ax2.set_ylabel('Air Temperature (K)', color=color2)
line2 = ax2.plot((tsa).sel(lat=slice(y1, y2), lon=slice(x1, x2)).mean(dim=['lon', 'lat']), color=color2, label='Air Temp')
ax2.tick_params(axis='y', labelcolor=color2)

# Create a combined legend for both lines
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='best')


plt.tight_layout()
plt.savefig('CP_diag.png',dpi=150)

NameError: name 'plt' is not defined

In [112]:
sw=sw.resample(time='1D').mean()